# SMITH regulatory-activity preservation

This notebook recomputes a representative analysis for the regulatory-activity Results section. It selects the strongest validated SMITH configuration for every dataset and panel size using the pinned five-run summary, then compares preservation of cell identity and developmental time.

[Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/regulatory_section/02_SMITH_Regulatory_Activity_source.ipynb)

## Setup

Run this notebook from a cloned SMITH repository with `pip install -e '.[notebooks]'`. All inputs are checksum-validated before analysis.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repository(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside a SMITH repository checkout.")


ROOT = find_repository(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))

from smith.reproducibility import check_case, load_cases, run_case

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
print(f"Repository: {ROOT}")


In [ ]:
CASE_ID = "02_regulatory_activity"
case = load_cases()[CASE_ID]
status = check_case(case)
if status["inputs"]:
    display(pd.DataFrame(status["inputs"])[["path", "exists", "sha256_ok"]])
else:
    print("This tutorial creates its deterministic input during execution.")
assert status["ready"], "The pinned tutorial inputs are missing or have changed."

output_dir = ROOT / "outputs" / "notebooks" / CASE_ID
result = run_case(case, output_dir)
print(f"Summary written to: {result['summary_json']}")
result


## Analysis

In [ ]:
best = pd.DataFrame(result["best_smith_by_dataset_and_panel"])
display(best.sort_values(["dataset", "panel_size"]))

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for dataset, group in best.groupby("dataset"):
    ax.scatter(group["celltype_accuracy"], group["time_pearson"], s=65, label=dataset)
    for row in group.itertuples():
        ax.annotate(str(row.panel_size), (row.celltype_accuracy, row.time_pearson),
                    xytext=(4, 3), textcoords="offset points", fontsize=8)
ax.set(xlabel="Cell-type kNN accuracy", ylabel="Developmental-time Pearson r",
       title="Validated SMITH configurations")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## What this reproduces

The table and plot recompute the joint cell-identity/developmental-time comparison from pinned aggregate results. Full Figure 3 regeneration additionally needs lineage-aware splits, TF/miRNA activity inference, baseline training, module coverage, co-activity reconstruction, and scRNA-to-TF transfer.